> Этот набор данных о показах фильмов в кинотеатрах состоит из 4 файлов .csv 
У нас есть таблица фактов `screenings`, таблицы измерений `movies, theaters` и таблица подизмерений  `genres`.

In [19]:
import pandas as pd
import duckdb
import etl
import plotly.express as px
import ddl

In [63]:
genres = etl.load_data('genres')
movies = etl.load_data('movies')
screenings = etl.load_data('screenings')
theaters = etl.load_data('theaters')

In [81]:
genres.sample(5)

,genre_id,genre_name
2,3,Action|Drama|War
8,9,Drama|Romance
17,18,Horror|Mystery|Thriller
20,21,Comedy|Romance
0,1,Comedy|Drama


>Эта таблица содержит информацию о жанрах фильмов. 
- `genre_id`: Используется как ключ для связывания жанров с другими таблицами
- `genre_name`: Название жанра или комбинация жанров, разделенная символом |

In [78]:
movies.sample(5)

,movie_id,movie_title,release_year,genre_id,imdb_rating,rotten_tomatoes,metacritic_rating
37,38,It's a Bird,1993,4,4.4,71,72.0
454,455,Tycoon (Oligarkh),2011,9,8.7,86,90.0
290,291,Jönssonligans största kupp,2012,12,5.6,48,87.0
52,53,"Conformist, The (Conformista, Il)",2005,16,4.8,83,91.0
485,486,Hamlet 2,2010,14,8.4,91,65.0


>Эта таблица содержит данные о фильмах. Каждая строка представляет собой запись об одном фильме, включая его основные характеристики
- `movie_id`: Идентификатор фильма для связи с другими таблицами
- `movie_title`:Название фильма.
- `release_year`: Год выпуска фильма.
- `genre_id`: Ссылка на идентификатор жанра из таблицы жанров (genre_id в таблице жанров). Определяет, к какому жанру или комбинации жанров относится фильм.
- `imdb_rating`:Рейтинг фильма на платформе IMDb.
Указывается в шкале от 0 до 10.
- `rotten_tomatoes`:Процент положительных отзывов на платформе Rotten Tomatoes.
Значение в диапазоне от 0 до 100.
- `metacritic_rating`: Рейтинг фильма на платформе Metacritic.
Значение в диапазоне от 0 до 100.


In [79]:
screenings.sample(5)

,screening_date,theater_id,movie_id,screening_time,tickets_sold,revenue
1369,2024,57,131,15:39:00,427,5498
533,2024,52,21,15:16:00,296,3572
2197,2024,32,416,12:43:00,325,5454
1771,2024,48,531,16:10:00,158,2040
1729,2024,14,577,20:51:00,102,3930


> Эта таблица содержит данные о сеансах фильмов. Каждая строка описывает конкретный показ фильма в определенный день.
- `screening_date`: Дата показа фильма.
- `theater_id`:Уникальный идентификатор кинотеатра.
Связывает сеанс с определенным местом проведения.
- `movie_id`: Уникальный идентификатор фильма из таблицы фильмов.
Позволяет определить, какой фильм показывался на данном сеансе.
- `screening_time`:Время начала сеанса.
- `tickets_sold`: Количество проданных билетов на данный сеанс.
- `revenue`: Общий доход от продажи билетов за данный сеанс.

In [80]:
theaters.sample(5)

,theater_id,theater_name,address
41,42,Feedmix,224 Parkside Road
64,65,Tagopia,035 Hallows Avenue
32,33,Wordtune,76 Novick Place
54,55,Skinder,56116 Muir Circle
51,52,Gigashots Theater,04162 Bunting Alley


>Эта таблица содержит информацию о кинотеатрах. Каждая строка представляет данные об одном кинотеатре
- `theater_id`: Уникальный идентификатор кинотеатра.
- `theater_name`:Название кинотеатра.
- `address`: Адрес кинотеатра.

In [15]:
dataframes = {
   'genres': genres ,
'movies' : movies,
'screenings' : screenings,
'theaters' : theaters
}

In [17]:
for name, df in dataframes.items():
    print('Таблица:', name)
    print('Размер таблицы', df.shape),
    print('---' * 10)

Таблица: genres
Размер таблицы (24, 2)
------------------------------
Таблица: movies
Размер таблицы (582, 7)
------------------------------
Таблица: screenings
Размер таблицы (3226, 6)
------------------------------
Таблица: theaters
Размер таблицы (84, 3)
------------------------------


In [16]:
for name, df in dataframes.items():
    print('Таблица', name)
    print('--'*16)
    print(df.isnull().sum())
    print(' ')
    print(' ')

Таблица genres
--------------------------------
genre_id      0
genre_name    0
dtype: int64
 
 
Таблица movies
--------------------------------
movie_id              0
movie_title           0
release_year          0
genre_id              0
imdb_rating          26
rotten_tomatoes       0
metacritic_rating    75
dtype: int64
 
 
Таблица screenings
--------------------------------
screening_date    0
theater_id        0
movie_id          0
screening_time    0
tickets_sold      0
revenue           0
dtype: int64
 
 
Таблица theaters
--------------------------------
theater_id      0
theater_name    0
address         0
dtype: int64
 
 


In [18]:
movies[movies['imdb_rating'].isnull()]

,movie_id,movie_title,release_year,genre_id,imdb_rating,rotten_tomatoes,metacritic_rating
11,12,I Know Who Killed Me,2008,24,NaN,69,76.0
18,19,Red Tails,2007,3,NaN,52,73.0
33,34,Undisputed,1990,23,NaN,56,73.0
45,46,Schultze Gets the Blues,1991,5,NaN,63,NaN
46,47,Testament,1999,20,NaN,86,72.0
94,95,Dirty Harry,1996,15,NaN,37,69.0
99,100,Airplane II: The Sequel,2013,4,NaN,61,91.0
129,130,Terminator 3: Rise of the Machines,2005,4,NaN,98,93.0
137,138,Act Da Fool,2004,3,NaN,73,65.0
144,145,Long Day's Journey Into Night,2003,18,NaN,55,72.0


In [20]:
fig = px.histogram(
    movies,
    x='imdb_rating',
    title='Распределение imdb_rating',
    labels={'imdb_rating': 'imdb_rating'}
)
fig.show()

In [24]:
numeric_columns = movies.select_dtypes(include=['number']).columns

# Вычисляем корреляционную матрицу для числовых столбцов
corr_matrix = movies[numeric_columns].corr()

# Строим тепловую карту
fig = px.imshow(
    corr_matrix,
    title='Корреляция между переменными',
    labels={'x': 'Переменные', 'y': 'Переменные'},
    color_continuous_scale='RdBu_r',
    zmin=-1, zmax=1
)

fig.show()

In [28]:
avg_rating_by_genre = movies.groupby('genre_id')['imdb_rating'].mean().reset_index()

fig = px.bar(
    avg_rating_by_genre,
    x='genre_id',
    y='imdb_rating',
    title='Средний IMDb рейтинг по жанрам',
    labels={'imdb_rating': 'Средний IMDb рейтинг', 'genre_id': 'Жанр'}
)

fig.show()

In [33]:
mean_imdb_rating = round(movies['imdb_rating'].mean(), 1)
movies['imdb_rating'].fillna(mean_imdb_rating, inplace=True)

Наиболее распространенный цвет: 7.1


C:\Users\user\AppData\Local\Temp\ipykernel_20276\2001408391.py:4: FutureWarning:

A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.





In [34]:
movies[movies['imdb_rating'].isnull()]

,movie_id,movie_title,release_year,genre_id,imdb_rating,rotten_tomatoes,metacritic_rating


In [36]:
movies[movies['metacritic_rating'].isnull()].sample(5)

,movie_id,movie_title,release_year,genre_id,imdb_rating,rotten_tomatoes,metacritic_rating
272,273,"Dead, The",2007,16,5.6,34,NaN
181,182,Flood,2008,12,7.1,89,NaN
295,296,"Amityville Curse, The",1998,12,6.9,46,NaN
160,161,Design for Living,1992,5,4.1,67,NaN
307,308,It's My Party,2005,7,7.1,81,NaN


In [44]:
fig = px.histogram(
    movies,
    x='metacritic_rating',
    title='Распределение metacritic_rating',
    labels={'metacritic_rating': 'metacritic_rating'}
)
fig.show()

In [38]:
avg_rating_by_genre = movies.groupby('genre_id')['metacritic_rating'].mean().reset_index()

# Строим график
fig = px.bar(
    avg_rating_by_genre,
    x='genre_id',
    y='metacritic_rating',
    title='Средний IMDb рейтинг по жанрам',
    labels={'metacritic_rating': 'Средний metacritic рейтинг', 'genre_id': 'Жанр'}
)

fig.show()

In [43]:
avg_rating_by_genre.sort_values(by = 'metacritic_rating')

,genre_id,metacritic_rating
6,7,65.266667
10,11,73.600000
13,14,74.190476
19,20,74.277778
20,21,74.809524
14,15,75.000000
21,22,75.280000
17,18,76.000000
0,1,76.307692
12,13,76.652174


In [50]:
# avg_rating_by_genre = movies.groupby('genre_id')['imdb_rating'].reset_index()
mode_prices = movies.groupby('genre_id')['metacritic_rating'].agg(lambda x: x.mode()[0])
mode_prices

genre_id
1     71.0
2     87.0
3     78.0
4     72.0
5     69.0
6     61.0
7     57.0
8     70.0
9     95.0
10    90.0
11    58.0
12    59.0
13    74.0
14    65.0
15    58.0
16    65.0
17    61.0
18    60.0
19    78.0
20    57.0
21    57.0
22    68.0
23    68.0
24    72.0
Name: metacritic_rating, dtype: float64

In [54]:
def replace_na_with_mode(row):
    if pd.isna(row['metacritic_rating']):
        return mode_prices[row['genre_id']]
    return row['metacritic_rating']


movies['metacritic_rating'] = movies.apply(replace_na_with_mode, axis=1)

In [55]:
movies[movies['metacritic_rating'].isnull()]

,movie_id,movie_title,release_year,genre_id,imdb_rating,rotten_tomatoes,metacritic_rating
